In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import glob
NPZ_PATHS = sorted(glob.glob("../data/npz_shards/all_turn_castle_ep_material_shard*.npz"))

BATCH_SIZE = 256
VAL_FRAC   = 0.1
CKPT_PATH  = "../weights/direct_cp_weights.pth"

INPUT_DIM = 28 * 8 * 8


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
import glob, os
print("cwd:", os.getcwd())
print("matches:", glob.glob("../data/npz_shards/*.npz"))
print("matches2:", glob.glob("../data/**/*.npz", recursive=True)[:20])


cwd: c:\Users\samue\OneDrive\Desktop\projects\ML\chess-learn\n5\training
matches: ['../data/npz_shards\\all_turn_castle_ep_material_shard0000.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0001.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0002.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0003.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0004.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0005.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0006.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0007.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0008.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0009.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0010.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0011.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0012.npz', '../data/npz_shards\\all_turn_castle_ep_material_shard0013.npz', '../dat

In [3]:
#-----------------------------
# Combine + create data set funcs/classes

import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset

def compute_global_max_abs_cp(paths):
    max_abs_cp = 0.0
    for p in paths:
        with np.load(p, mmap_mode='r') as zf:
            Zcp = zf['evaluations']
            max_abs_cp = max(max_abs_cp, float(np.max(np.abs(Zcp))))
    return max_abs_cp

class ChessEvalNPZ(Dataset):
    def __init__(self, npz_path: str, flatten: bool = True, max_abs_cp: float = 13000.0):
        self.path = npz_path
        self.flatten = flatten
        with np.load(self.path, mmap_mode='r') as zf:
            self.N = int(zf['evaluations'].shape[0])
        self.M_cp = float(max_abs_cp)
        self._X = None
        self._Zcp = None

    def _ensure_open(self):
        if self._X is None:
            zf = np.load(self.path, mmap_mode='r')
            self._X = zf['positions']      # (N,28,8,8)
            self._Zcp = zf['evaluations']  # (N,)

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        self._ensure_open()
        x = self._X[idx].astype(np.float32)
        if self.flatten:
            x = x.reshape(-1)
        z_cp = float(self._Zcp[idx])
        cp = np.clip(z_cp, -self.M_cp, self.M_cp)
        return torch.from_numpy(x), torch.tensor([cp], dtype=torch.float32)




In [21]:
# -----------------------------
# Building loaders



#global_M = compute_global_max_abs_cp(NPZ_PATHS)
global_M = 13000.0
datasets = [ChessEvalNPZ(p, flatten=True, max_abs_cp=global_M) for p in NPZ_PATHS]
full_ds = ConcatDataset(datasets)
N = len(full_ds)
val_len = int(N * VAL_FRAC)
train_len = N - val_len

g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_ds, [train_len, val_len], generator=g)

NUM_WORKERS = 0  # You can try 2–4 if CPU allows; dataset is worker-safe
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Dataset: N={N} | input_dim={INPUT_DIM} | train={train_len} | val={val_len}")
xb0, yb0 = next(iter(train_loader))
print("One sample check:", xb0.shape)  # (B, 796)

Dataset: N=16586527 | input_dim=1792 | train=14927875 | val=1658652
One sample check: torch.Size([256, 1792])


In [22]:
print(xb0.shape)


torch.Size([256, 1792])


In [23]:
import torch
import numpy as np
from collections import Counter

# define bins and labels for reporting
edges = np.array([0,150,600,1200,1e9])  # [0–150], (150–600], (600–1200], >1200
labels = ["=0..150", "150..600", "600..1200", ">1200"]

def bin_counts(cp):
    cp = cp.abs().cpu().numpy().reshape(-1)
    idx = np.digitize(cp, edges) - 1
    c = Counter(idx)
    return np.array([c.get(i,0) for i in range(len(labels))])

def inspect_loader(loader, n_batches=20):
    tot = np.zeros(len(labels), dtype=int)
    n = 0
    for xb, yb in loader:
        tot += bin_counts(yb.squeeze(1))
        n += 1
        if n >= n_batches:
            break
    share = tot / tot.sum()
    report = {lab: float(s) for lab, s in zip(labels, share)}
    return tot, report

# Example usage:
tot_s, rep_s = inspect_loader(train_loader, n_batches=50)
print("Sample counts over 50 batches:", tot_s)
print("Proportions:", rep_s)

# Sample counts over 50 batches: [9024 3449  315   12]
# Proportions: {'=0..150': 0.705, '150..600': 0.269453125, '600..1200': 0.024609375, '>1200': 0.0009375}
# Sample counts over 50 batches: [6495 4723 1417  165]
# Proportions: {'=0..150': 0.507421875, '150..600': 0.368984375, '600..1200': 0.110703125, '>1200': 0.012890625}


Sample counts over 50 batches: [6018 4507 1422  853]
Proportions: {'=0..150': 0.47015625, '150..600': 0.352109375, '600..1200': 0.11109375, '>1200': 0.066640625}


In [24]:
# a) shapes & dtypes
print("xb0:", xb0.shape, xb0.dtype, "yb0:", yb0.shape, yb0.dtype)

# b) value ranges
xb_min, xb_max = float(xb0.min()), float(xb0.max())
print(f"X range first batch: [{xb_min}, {xb_max}]")

# c) targets in [0,1] after transform?
print("y batch stats: min=", float(yb0.min()), "max=", float(yb0.max()))

# d) no NaNs/inf in a couple random batches
def has_bad(t): 
    return torch.isnan(t).any().item() or torch.isinf(t).any().item()

for i, (xchk, ychk) in enumerate(train_loader):
    if i == 3: break
    assert not has_bad(xchk), "NaN/Inf in inputs"
    assert not has_bad(ychk), "NaN/Inf in targets"
print("Basic data checks passed.")


xb0: torch.Size([256, 1792]) torch.float32 yb0: torch.Size([256, 1]) torch.float32
X range first batch: [-9.0, 10.0]
y batch stats: min= -13000.0 max= 13000.0
Basic data checks passed.


In [25]:
def v_to_cp(v, M):   # v in R (not necessarily clipped); returns cp
    return M * (2.0 * v - 1.0)

v_to_cp(0.10965852439403534, global_M), global_M

(-10148.878365755081, 13000.0)

In [26]:
#Check the extremes exist in entire data

# all_y = []
# for _, y in train_loader:
#     all_y.append(y)
# ycat = torch.cat(all_y)
# print("Global target min/max:", float(ycat.min()), float(ycat.max()))


In [27]:
#More config
import random, numpy as np, torch

LR         = 1e-3
EPOCHS     = 100
PATIENCE   = 8
HLAYER_SIZES = (2048, 2048, 1024)
SEED = 42


random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [28]:
# -----------------------------
# Model

class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden, p_drop=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h, bias=True),
                nn.BatchNorm1d(h),
                nn.ELU(inplace=True),
                nn.Dropout(p_drop),
            ]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):  # (B, 896)
        return self.net(x)

In [29]:
import torch.nn as nn
import torch.nn.functional as F

class CenterWeightedHuber(nn.Module):
    def __init__(self, delta=150.0, clip=3000.0, k=400.0, min_w=0.5):
        super().__init__()
        self.delta = float(delta)
        self.clip  = float(clip)
        self.k     = float(k)
        self.min_w = float(min_w)

    def forward(self, pred_cp, true_cp):
        if pred_cp.dim() > 1 and pred_cp.size(-1) == 1:
            pred_cp = pred_cp.squeeze(-1)
        if true_cp.dim() > 1 and true_cp.size(-1) == 1:
            true_cp = true_cp.squeeze(-1)

        t = true_cp.clamp(-self.clip, self.clip)
        base = F.smooth_l1_loss(pred_cp, t, beta=self.delta, reduction='none')

        # emphasize near-equal positions
        w = torch.exp(- (t.abs()/self.k)**2)
        w = self.min_w + (1 - self.min_w) * w

        return (base * w).mean()



In [30]:
model = MLP(INPUT_DIM, HLAYER_SIZES).to(device)

criterion = CenterWeightedHuber(delta=150, k=400, min_w=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.99), eps=1e-8)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)
def count(x):
    f = 0
    for i in range(len(x)-1):
        f += x[i]*x[i+1]
    return f + sum(x[1:])


Nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {Nparams/1e6:.3f}M")


Params: 9.978M


In [31]:
model.train()
xb, yb = xb0.to(device), yb0.to(device)
out = model(xb)
print("forward:", out.shape)
loss = criterion(out, yb)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
optimizer.step(); optimizer.zero_grad(set_to_none=True)
print("one-step OK, loss:", float(loss))


forward: torch.Size([256, 1])
one-step OK, loss: 231.814697265625


In [15]:
#Confirm net is big enough to overfit if i tried

from torch.utils.data import Subset
small_idx = list(range(min(1024, len(train_ds))))
tiny_loader = DataLoader(Subset(train_ds, small_idx), batch_size=256, shuffle=True)
model_small = MLP(INPUT_DIM, HLAYER_SIZES).to(device)
opt_small  = torch.optim.Adam(model_small.parameters(), lr=1e-3, betas=(0.9,0.99), eps=1e-8)
crit = CenterWeightedHuber(delta=150, k=400, min_w=0.5)
for e in range(5000):
    tl=0.0
    for xb, yb in tiny_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_small.zero_grad(set_to_none=True)
        pred = model_small(xb)
        loss = crit(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_small.parameters(), 5.0)
        opt_small.step()
        tl += loss.item()*xb.size(0)
    print(f"(tiny) epoch {e+1} loss {tl/len(small_idx):.6f}")


(tiny) epoch 1 loss 177.809475
(tiny) epoch 2 loss 176.049789
(tiny) epoch 3 loss 174.115330
(tiny) epoch 4 loss 172.640911
(tiny) epoch 5 loss 171.270973
(tiny) epoch 6 loss 169.796307
(tiny) epoch 7 loss 167.942429
(tiny) epoch 8 loss 166.297489
(tiny) epoch 9 loss 164.110409
(tiny) epoch 10 loss 162.225670
(tiny) epoch 11 loss 160.712584
(tiny) epoch 12 loss 158.365330
(tiny) epoch 13 loss 156.408974
(tiny) epoch 14 loss 154.621719
(tiny) epoch 15 loss 153.411972
(tiny) epoch 16 loss 151.536148
(tiny) epoch 17 loss 150.083389
(tiny) epoch 18 loss 148.138615
(tiny) epoch 19 loss 147.282063
(tiny) epoch 20 loss 146.020374
(tiny) epoch 21 loss 145.653271
(tiny) epoch 22 loss 145.389397
(tiny) epoch 23 loss 144.050755
(tiny) epoch 24 loss 142.252768
(tiny) epoch 25 loss 139.775005
(tiny) epoch 26 loss 138.452217
(tiny) epoch 27 loss 137.200108
(tiny) epoch 28 loss 136.356657
(tiny) epoch 29 loss 135.816679
(tiny) epoch 30 loss 134.580935
(tiny) epoch 31 loss 134.078629
(tiny) epoch 32 l

In [32]:
# -----------------------------
# Train loop + early stopping
# -----------------------------
best_val = float('inf')
pat = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) #try 2.0
        optimizer.step()
        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * xb.size(0)

    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d} | train MSE: {train_loss:.6f} | val MSE: {val_loss:.6f}")

    if val_loss < best_val - 1e-6:
        best_val = val_loss
        pat = 0
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "global_M": float(global_M),
            "input_dim": int(INPUT_DIM),
            "arch": list(HLAYER_SIZES),
                    }, CKPT_PATH)

    else:
        pat += 1
        if pat >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val MSE: {best_val:.6f}")
            break

print("Best val MSE:", best_val)
print(f"Saved best model to: {CKPT_PATH}")

Epoch 01 | train MSE: 153.731686 | val MSE: 144.158504
Epoch 02 | train MSE: 142.460423 | val MSE: 144.321169
Epoch 03 | train MSE: 137.144742 | val MSE: 132.539011
Epoch 04 | train MSE: 133.392481 | val MSE: 128.131271
Epoch 05 | train MSE: 130.448831 | val MSE: 127.450840
Epoch 06 | train MSE: 128.060718 | val MSE: 123.063069
Epoch 07 | train MSE: 125.972598 | val MSE: 123.610472
Epoch 08 | train MSE: 124.200298 | val MSE: 122.159136
Epoch 09 | train MSE: 122.598177 | val MSE: 118.433386
Epoch 10 | train MSE: 121.151053 | val MSE: 118.488205
Epoch 11 | train MSE: 119.821257 | val MSE: 117.612330
Epoch 12 | train MSE: 118.652564 | val MSE: 113.951800
Epoch 13 | train MSE: 117.534504 | val MSE: 113.761124
Epoch 14 | train MSE: 116.506085 | val MSE: 117.258993
Epoch 15 | train MSE: 115.523093 | val MSE: 112.599215
Epoch 16 | train MSE: 114.648329 | val MSE: 111.740253
Epoch 17 | train MSE: 113.829913 | val MSE: 110.473184
Epoch 18 | train MSE: 113.028513 | val MSE: 110.199893
Epoch 19 |

KeyboardInterrupt: 

In [17]:
print("N train:", len(train_loader.dataset))
print("batches/epoch:", len(train_loader))
print("model device:", next(model.parameters()).device)


N train: 14927875
batches/epoch: 58313
model device: cuda:0


In [ ]:
def evaluate_mse(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = model(xb)
            total += criterion(pred, yb).item() * xb.size(0)
            n += xb.size(0)
    return total / n

final_val_mse = evaluate_mse(model, val_loader)
print("Final (reloaded) val MSE:", final_val_mse)

NameError: name 'model' is not defined

In [ ]:
#not big enough to overfit
#need additional info
#look at tweaking params

In [18]:
print("global_M:", global_M)


global_M: 100000.0


In [34]:
mins = []
maxs = []

for _, y in train_loader:
    mins.append(float(y.min()))
    maxs.append(float(y.max()))
    if len(mins) == 10000:
        break

print("Observed clipped y range:",
      min(mins),
      max(maxs),
      " | expected:",
      -global_M,
      global_M)


Observed clipped y range: -13000.0 13000.0  | expected: -13000.0 13000.0
